In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install akshare tensorflow scikit-learn matplotlib

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.3 MB/s eta 0:00:00
  Created wheel for jsonpath: filename=jsonpath-0.82.2-py3-none-any.whl size=5615 sha256=43d8e4f04fb6238d0c44979102a8f5c0fc6de2c548c781f754c548a3f09da7c4
  Stored in directory: /root/.cache/pip/wheels/73/76/e2/980a29341fe37a583ada29594ed529708d5e8e2c0f9d97c3cc
Successfully built jsonpath


In [3]:
# ✅ 导入必要库
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import zipfile
from tensorflow.keras.models import Model, load_model # 导入 load_model 函数
from tensorflow.keras.layers import Input, MultiHeadAttention, LayerNormalization, Dense, Dropout, TimeDistributed, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy
from tensorflow.keras.metrics import MeanAbsoluteError, BinaryAccuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from keras.callbacks import ModelCheckpoint, Callback
from tqdm.keras import TqdmCallback


# ✅ 设置股票代码与时间区间
stock_codes = ['601138','002269']
start_date = "2024-09-30"
end_date = "2025-09-30"
save_path = "./data/"
os.makedirs(save_path, exist_ok=True)

# ✅ 标准化字段名
def standardize_columns(df):
    column_mapping = {
        "日期": "Date", "股票代码": "Code", "开盘": "Open", "收盘": "Close",
        "最高": "High", "最低": "Low", "成交量": "Volume", "成交额": "Amount",
        "振幅": "Amplitude", "涨跌幅": "ChangePct", "涨跌额": "ChangeAmt", "换手率": "Turnover",
    }
    for col in df.columns:
        if col not in column_mapping:
            column_mapping[col] = col
    valid_column_mapping = {old: new for old, new in column_mapping.items() if old in df.columns}
    df.rename(columns=valid_column_mapping, inplace=True)
    return df

# ✅ 计算 RSI 指标（我们自己计算需要的指标）引用股票知识
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(window).mean()
    loss = -delta.where(delta < 0, 0).rolling(window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# ✅ 特征工程与目标构造
def process_features(df):
    if df is None or df.empty:
        return None, None, None, None, None, None

    df["涨跌幅"] = df.get("ChangePct", 0)
    df["涨跌方向"] = (df["涨跌幅"] > 0).astype(int)

    # 技术指标构造
    df["MA5"] = df["Close"].rolling(5).mean()
    df["MA10"] = df["Close"].rolling(10).mean()
    df["EMA12"] = df["Close"].ewm(span=12, adjust=False).mean()
    df["EMA26"] = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = df["EMA12"] - df["EMA26"]
    df["RSI"] = compute_rsi(df["Close"])
    df["Return_1d"] = df["Close"].pct_change()
    df["Volatility_5d"] = df["Close"].rolling(5).std()
    df["Weekday"] = df.index.weekday
    df["IsMonthEnd"] = df.index.is_month_end.astype(int)
    df["MoneyFlow"] = df.apply(lambda row: row["Amount"] / row["Volume"] if row["Volume"] != 0 else 0, axis=1)

    # 资金流衍生特征
    if "MainNetInflow" in df.columns and "MainInflowRatio" in df.columns:
        df["MainNetInflow_Smooth"] = df["MainNetInflow"].ewm(span=5, adjust=False).mean()
        df["MainNetInflow_Normalized"] = (df["MainNetInflow"] - df["MainNetInflow"].mean()) / df["MainNetInflow"].std()
        # Use .fillna(0) or .dropna() as pct_change can produce NaNs
        df["MainNetInflow_Ratio_Change"] = df["MainInflowRatio"].pct_change().fillna(0)
        df["MainNetInflow_5d_Std"] = df["MainNetInflow"].rolling(window=5).std()
    else:
        # Initialize these columns with 0 if main funds flow data is missing
        df["MainNetInflow_Smooth"] = 0
        df["MainNetInflow_Normalized"] = 0
        df["MainNetInflow_Ratio_Change"] = 0
        df["MainNetInflow_5d_Std"] = 0
        print("⚠️ Funds flow columns not found, initializing derivative features to 0.")


    # 异动信号（涨停/跌停/资金突变）
    # 涨停标记：假设涨停是接近最高价且涨幅较大
    df["IsLimitUp"] = ((df["High"] >= df["Close"] * 0.99) & (df["ChangePct"] > 9.5)).astype(int)
    # 跌停标记：假设跌停是接近最低价且跌幅较大
    df["IsLimitDown"] = ((df["Low"] <= df["Close"] * 1.01) & (df["ChangePct"] < -9.5)).astype(int)

    # 连续涨停次数 (Simplified: count consecutive IsLimitUp)
    # This requires a more sophisticated approach for true consecutive counting,
    # but a simple rolling sum can indicate recent frequency.
    df["ConsecutiveLimitUp_5d"] = df["IsLimitUp"].rolling(window=5).sum()

    # 主力资金流入突变：资金净流入 > 均值 + 2σ (requires MainNetInflow)
    if "MainNetInflow" in df.columns:
        mean_inflow = df["MainNetInflow"].mean()
        std_inflow = df["MainNetInflow"].std()
        df["IsAbnormalInflow"] = (df["MainNetInflow"] > mean_inflow + 2 * std_inflow).astype(int)
    else:
        df["IsAbnormalInflow"] = 0
        print("⚠️ MainNetInflow column not found, skipping IsAbnormalInflow feature.")

    # 换手率激增：换手率 > 历史均值 + 3σ
    mean_turnover = df["Turnover"].mean()
    std_turnover = df["Turnover"].std()
    df["IsTurnoverSpike"] = (df["Turnover"] > mean_turnover + 3 * std_turnover).astype(int)


    df.dropna(inplace=True)

    # 特征与目标字段
    features = [
        "Open", "High", "Low", "Close", "Volume", "Amount", "Turnover",
        "MA5", "MA10", "EMA12", "EMA26", "MACD", "RSI",
        "Return_1d", "Volatility_5d", "Weekday", "IsMonthEnd", "MoneyFlow", "ChangePct",
        "MainNetInflow", "MainInflowRatio",
        "MainNetInflow_Smooth", "MainNetInflow_Normalized",
        "MainNetInflow_Ratio_Change", "MainNetInflow_5d_Std",
        "IsLimitUp", "IsLimitDown", "ConsecutiveLimitUp_5d",
        "IsAbnormalInflow", "IsTurnoverSpike" # Add new features here
    ]

    # Ensure only existing columns are included in features
    features = [col for col in features if col in df.columns]


    targets_regression = ["Close", "涨跌幅"] # Keep only 'Close' and '涨跌幅' for now, '主力净流入' needs careful handling of scaling and inverse transform
    targets_regression = [t for t in targets_regression if t in df.columns]
    target_classification = "涨跌方向"

    # Ensure target_classification exists
    if target_classification not in df.columns:
        print(f"❌ Classification target '{target_classification}' not found after processing.")
        return None, None, None, None, None, None


    cols_to_scale = [col for col in features + targets_regression if col in df.columns and df[col].dtype in ['int64', 'float64']]
    scaler = MinMaxScaler()
    df_scaled = scaler.fit_transform(df[cols_to_scale])

    # Re-align target_regression_data and target_classification_data with the scaled data
    # Ensure the order of columns in targets_regression matches the order they appear in cols_to_scale that are also in targets_regression
    scaled_cols_in_targets_regression = [col for col in cols_to_scale if col in targets_regression]
    target_regression_data = df[scaled_cols_in_targets_regression].values

    target_classification_data = df[target_classification].values
    actual_regression_target_names = scaled_cols_in_targets_regression # Use the actual columns used for regression

    return df, df_scaled, target_regression_data, target_classification_data, scaler, actual_regression_target_names


# ✅ 加载日线与资金流数据
def load_daily_and_fundflow(code, backtest_start, backtest_end):
    prefix = code[-6:]
    market_prefix = "sz" if code.startswith("0") else "sh"
    daily_path = f"/content/drive/MyDrive/Colab Notebooks/ant_data/kline/{prefix}_daily.csv"
    fundflow_path = f"/content/drive/MyDrive/Colab Notebooks/ant_data/jiqi_do/{market_prefix}{code}.csv"

    if not os.path.exists(daily_path):
        print(f"⚠️ 缺失日线数据文件：{daily_path}")
        return [None] * 6

    try:
        df = pd.read_csv(daily_path)
        df = standardize_columns(df.copy())
        if "Date" not in df.columns:
            print(f"❌ 'Date'列缺失：{daily_path}")
            return [None] * 6

        df["Date"] = pd.to_datetime(df["Date"])
        df.set_index("Date", inplace=True)
        df.sort_index(inplace=True)
        df = df[~df.index.duplicated(keep='last')]
        # Removed date slicing here: df = df.loc[pd.to_datetime(backtest_start):pd.to_datetime(backtest_end)].copy()

        if os.path.exists(fundflow_path):#检查路径 fundflow_path 是否存在，避免文件不存在时报错。
            df_fund = pd.read_csv(fundflow_path) #读取 CSV 文件为 DataFrame。
            fundflow_column_mapping = {
                "opendate": "Date",
                "netamount": "MainNetInflow",
                "ratioamount": "MainInflowRatio"
            }  #改为英文列名防环境报错
            df_fund.rename(columns=fundflow_column_mapping, inplace=True)
            df_fund.drop(columns=[col for col in ["turnover", "trade", "changeratio"] if col in df_fund.columns], inplace=True)
            # 删除一些不需要的列（如果存在），避免干扰后续处理。
            if "Date" in df_fund.columns:
                df_fund["Date"] = pd.to_datetime(df_fund["Date"])
                df_fund.set_index("Date", inplace=True)
                df = df.join(df_fund, how="left")
                print(f"💰 资金流数据合并成功：{code}")
            else:
                print(f"❌ 资金流数据缺少 'Date' 列：{code}")
        else:
            print(f"⚠️ 缺失资金流文件：{fundflow_path}")

        for col in ["MainNetInflow", "MainInflowRatio"]:
            if col not in df.columns:
                df[col] = 0
                print(f"⚠️ 初始化缺失资金流字段：{col}")

        # Apply date slicing after feature processing to ensure all data is available for indicators
        df = df.loc[pd.to_datetime(backtest_start):pd.to_datetime(backtest_end)].copy()
        if df.empty:
            print(f"⚠️ 切片后数据为空：{code}")
            return [None] * 6

        return process_features(df)

    except Exception as e:
        print(f"❌ 数据处理失败：{code} → {e}")
        return [None] * 6

# ✅ 构造滑窗样本（支持多窗口），修正分类目标形状
def create_sequences(data_scaled, target_regression_data, target_classification_data, window_size=60, pred_size=15):
    X, y_regression, y_classification = [], [], []
    if len(data_scaled) < window_size + pred_size:
        print(f"⚠️ 数据不足：需要至少 {window_size + pred_size} 条，但只有 {len(data_scaled)}")
        return np.array(X), np.array(y_regression), np.array(y_classification)

    for i in range(len(data_scaled) - window_size - pred_size + 1):
        X.append(data_scaled[i:i+window_size])
        y_regression.append(target_regression_data[i+window_size:i+window_size+pred_size])
        # 确保分类目标有最后一个维度为 1
        y_classification.append(target_classification_data[i+window_size:i+window_size+pred_size].reshape(-1, 1))

    # Check shape and add dimension if necessary
    y_classification_array = np.array(y_classification)
    # Check if y_classification_array is not empty before checking shape
    if y_classification_array.size > 0 and y_classification_array.ndim == 2:
      # This case should ideally not happen after the reshape above, but as a safeguard
      print("⚠️ Warning: y_classification_array is 2D, reshaping to 3D.")
      y_classification_array = y_classification_array.reshape(y_classification_array.shape[0], y_classification_array.shape[1], 1)


    return np.array(X), np.array(y_regression), y_classification_array

# ✅ 构建 Attention 模型（含残差连接与升维）
def build_transformer_model(input_shape, num_regression_targets, pred_size=15):
    inputs = Input(shape=input_shape)
    projected_inputs = Dense(64)(inputs)
    attn = MultiHeadAttention(num_heads=4, key_dim=32)(projected_inputs, projected_inputs)
    attn_out = LayerNormalization()(attn + projected_inputs)
    x = Dense(64, activation='relu')(attn_out)
    x = Dropout(0.2)(x)
    x = LayerNormalization()(x + attn_out)

    regression_output = TimeDistributed(Dense(num_regression_targets))(x)
    regression_output = Lambda(lambda t: t[:, -pred_size:, :], name="regression_output")(regression_output)

    classification_output = TimeDistributed(Dense(1, activation='sigmoid'))(x)
    classification_output = Lambda(lambda t: t[:, -pred_size:, :], name="classification_output")(classification_output)

    model = Model(inputs=inputs, outputs=[regression_output, classification_output])
    model.compile(optimizer=Adam(),
                  loss={"regression_output": MeanSquaredError(), "classification_output": BinaryCrossentropy()},
                  metrics={"regression_output": MeanAbsoluteError(), "classification_output": BinaryAccuracy()})
    return model

# ✅ 训练模型（支持动态轮数、断点续训与进度条）
def train_transformer_model(X_train, y_train_regression, y_train_classification, input_shape, model_path, epochs=50):
    num_regression_targets = y_train_regression.shape[-1]
    pred_size = y_train_regression.shape[1]

    # Check if the model file exists in the local path (not Drive)
    # For this specific modification, we'll assume training from scratch or loading from local
    # The backtesting loop will handle Drive loading/saving.
    # If you want to load from Drive here, you would need to adjust the model_path logic.
    model = build_transformer_model(input_shape, num_regression_targets, pred_size)

    # Set ModelCheckpoint callback to save the best model (to local path if used here)
    # model_path needs to be in the drive for persistence
    # checkpoint_cb = ModelCheckpoint(model_path, save_best_only=True, monitor='val_loss', mode='min', verbose=0) # Disable this here

    # Set TqdmCallback for training progress bar
    tqdm_callback = TqdmCallback(verbose=2)

    # Train the model, including callbacks
    history = model.fit(X_train,
                        {"regression_output": y_train_regression, "classification_output": y_train_classification},
                        epochs=epochs,
                        batch_size=32,
                        verbose=0, # Disable fit's verbose, use TqdmCallback instead
                        callbacks=[tqdm_callback]) # Only use tqdm_callback here
    return model

# ✅ 预测与图表输出（含累计收益与混淆矩阵）
def predict_and_plot_targets(model, df, scaled_data, scaler, window_size, stock_code, actual_regression_target_names, output_dir="./charts"):
    import matplotlib
    matplotlib.rcParams['font.sans-serif'] = ['SimHei']
    matplotlib.rcParams['axes.unicode_minus'] = False

    print(f"Attempting to plot for {stock_code} with window {window_size} and output_dir {output_dir}")

    if len(scaled_data) < window_size:
        print(f"⚠️ 数据不足以创建回溯窗口 ({window_size})，跳过预测：{stock_code}")
        return

    last_window = scaled_data[-window_size:]
    # Ensure the data for prediction has the correct dimensions
    if last_window.shape[0] != window_size or last_window.shape[1] != scaled_data.shape[1]:
         print(f"❌ 创建预测窗口数据维度错误，跳过预测：{stock_code}")
         return

    try:
        predictions = model.predict(np.expand_dims(last_window, axis=0))
        regression_prediction = predictions[0][0]
        classification_prediction = predictions[1][0]
        print(f"Prediction shapes: Regression: {regression_prediction.shape}, Classification: {classification_prediction.shape}")
    except Exception as e:
        print(f"❌ 模型预测失败：{stock_code} → {e}")
        return


    num_regression_targets = regression_prediction.shape[-1]
    pred_size = regression_prediction.shape[0]
    print(f"Predicted pred_size: {pred_size}, num_regression_targets: {num_regression_targets}")

    # ✅ Inverse transform regression results
    # Create a dummy array with the same number of features as the scaler was fitted on
    dummy_inverse = np.zeros((pred_size, scaler.feature_names_in_.shape[0]))
    # Find the indices in the scaler's feature names that correspond to the regression targets
    try:
        target_indices = [np.where(scaler.feature_names_in_ == name)[0][0] for name in actual_regression_target_names]
        print(f"Target indices for inverse transform: {target_indices}")
    except IndexError:
        print(f"❌ 反标准化时目标列在 scaler 特征名称中未找到，跳过回归结果保存和绘图：{stock_code}")
        return

    if len(target_indices) != num_regression_targets:
         print(f"❌ 反标准化时目标列数量不匹配 ({len(target_indices)} vs {num_regression_targets})，跳过回归结果保存和绘图：{stock_code}")
    else:
        for i, idx in enumerate(target_indices):
            dummy_inverse[:, idx] = regression_prediction[:, i]
        recovered_full = scaler.inverse_transform(dummy_inverse)
        # Extract only the columns that were the regression targets
        recovered_regression_targets = recovered_full[:, target_indices]
        print(f"Shape of recovered_regression_targets: {recovered_regression_targets.shape}")


        # ✅ Construct prediction dates
        # Save plots to a local temporary directory instead of directly to Drive charts subdir
        local_charts_dir = f"./local_charts/window_{window_size}"
        os.makedirs(local_charts_dir, exist_ok=True)
        print(f"Saving plots to local directory: {local_charts_dir}")

        today = datetime.today().strftime('%Y-%m-%d')
        # Ensure df.index is not empty
        if df.index.empty:
            print(f"❌ DataFrame 索引为空，无法构造预测日期，跳过绘图和保存：{stock_code}")
            return
        # Use df.index[-1] to get the last date in the available data
        last_date_in_df = df.index[-1]
        prediction_dates = pd.date_range(start=last_date_in_df + pd.Timedelta(days=1), periods=pred_size, freq='B')
        print(f"Prediction dates generated: {prediction_dates.tolist()}")


        # ✅ Regression plots: Predicted vs Actual
        for i, name in enumerate(actual_regression_target_names):
            plt.figure(figsize=(10, 4))

            # Check if the current target requires dual Y-axis
            # Add other target names here if needed, but stick to 'Close' and '涨跌幅' as per the updated targets_regression
            if name in ["Close", "涨跌幅"]:
                # ✅ Dual Y-axis plot
                ax1 = plt.gca()
                ax1.plot(prediction_dates, recovered_regression_targets[:, i], marker='o', label=f"预测{name}", color='orange')
                ax1.set_ylabel(f"预测{name}", color='orange')
                ax1.tick_params(axis='y', labelcolor='orange')

                ax2 = ax1.twinx()
                # Plot actual data for the last pred_size days available in df
                if name in df.columns and len(df) >= pred_size:
                    # Ensure the dates for plotting actual data align with the last dates in df
                    actual_dates_for_plot = df.index[-pred_size:]
                    actual_values_for_plot = df[name].iloc[-pred_size:]
                    print(f"Plotting actual {name} data from {actual_dates_for_plot.tolist()}")
                    if len(actual_dates_for_plot) == len(actual_values_for_plot):
                         ax2.plot(actual_dates_for_plot, actual_values_for_plot, marker='x', linestyle='--', label=f"实际{name}", color='blue')
                         ax2.set_ylabel(f"实际{name}", color='blue')
                         ax2.tick_params(axis='y', labelcolor='blue')
                    else:
                         print(f"⚠️ 实际{name}数据 ({len(actual_values_for_plot)}) 与日期长度 ({len(actual_dates_for_plot)}) 不匹配，跳过在双Y轴图上绘制实际值：{stock_code}")
                else:
                     print(f"⚠️ 实际{name}数据不足或缺失，跳过在双Y轴图上绘制实际值：{stock_code}")

                plt.title(f"{stock_code} - 未来{pred_size}天{name}预测 vs 实际 (回归 - 双Y轴)")
                plt.xlabel("日期")
                # Combine legends from both axes
                lines_1, labels_1 = ax1.get_legend_handles_labels()
                lines_2, labels_2 = ax2.get_legend_handles_labels()
                ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

            else:
                # ✅ Single Y-axis plot for other regression targets
                plt.plot(prediction_dates, recovered_regression_targets[:, i], marker='o', label=f"预测{name}")
                if name in df.columns and len(df) >= pred_size: # Ensure df has enough rows for slicing
                    # Use actual data's index for plotting
                    actual_dates_for_plot = df.index[-pred_size:]
                    actual_values_for_plot = df[name].iloc[-pred_size:]
                    if len(actual_dates_for_plot) == len(actual_values_for_plot):
                         plt.plot(actual_dates_for_plot, actual_values_for_plot, marker='x', linestyle='--', label=f"实际{name}")
                plt.title(f"{stock_code} - 未来{pred_size}天{name}预测 (回归)")
                plt.xlabel("日期")
                plt.ylabel(name)
                plt.legend()

            plt.grid(True)
            plt.tight_layout()
            # Save to local charts directory
            chart_path = os.path.join(local_charts_dir, f"{stock_code}_{today}_{name}_regression.png")
            plt.savefig(chart_path)
            plt.close()
            print(f"✅ 回归图已保存到本地：{chart_path}")

        # ✅ Save regression predictions to CSV (to Google Drive)
        try:
            regression_df = pd.DataFrame(recovered_regression_targets, index=prediction_dates, columns=actual_regression_target_names)
            # Use the original output_dir (Drive path) for CSV
            regression_path = os.path.join(output_dir, f"{stock_code}_{today}_regression_predictions.csv")
            regression_df.to_csv(regression_path)
            print(f"✅ 回归预测CSV已保存至 Google Drive：{regression_path}")
        except Exception as e:
            print(f"❌ 回归CSV保存失败：{stock_code} → {e}")


    # ✅ Classification plot + Confusion Matrix
    predicted_direction = (classification_prediction > 0.5).astype(int)
    direction_labels = ['跌/平' if d == 0 else '涨' for d in predicted_direction]

    plt.figure(figsize=(10, 4))
    plt.plot(prediction_dates, predicted_direction, marker='o', label="预测涨跌方向")
    for i, label in enumerate(direction_labels):
        plt.text(prediction_dates[i], predicted_direction[i], label, ha='center', va='bottom')
    plt.title(f"{stock_code} - 未来{pred_size}天涨跌方向预测 (分类)")
    plt.xlabel("日期")
    plt.ylabel("方向 (0: 跌/平, 1: 涨)")
    plt.yticks([0, 1], ['跌/平', '涨'])
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    # Save to local charts directory
    chart_path = os.path.join(local_charts_dir, f"{stock_code}_{today}_direction_classification.png")
    plt.savefig(chart_path)
    plt.close()
    print(f"✅ 分类图已保存到本地：{chart_path}")


    # ✅ Confusion Matrix (using the most recent actual data)
    # Need actual classification data for the predicted dates from the backtest loop
    # This function is for plotting the *last* prediction. Confusion matrix for backtest
    # should be calculated from the aggregated results across all steps.
    # Skipping Confusion Matrix plot here, it will be calculated later from combined results.
    print("⚠️ Confusion Matrix calculation skipped in predict_and_plot_targets. Calculate from aggregated backtest results.")


    # ✅ Cumulative Returns Plot (using the predicted direction as a strategy)
    # Need actual close prices corresponding to the prediction dates for this
    # This function is for plotting the *last* prediction. Cumulative returns for backtest
    # should be calculated from the aggregated results across all steps.
    # Skipping Cumulative Returns plot here, it will be calculated later from combined results.
    print("⚠️ Cumulative Returns plot skipped in predict_and_plot_targets. Calculate from aggregated backtest results.")


    # ✅ Save classification predictions to CSV (to Google Drive)
    try:
        classification_df = pd.DataFrame(
            np.column_stack([
                predicted_direction.flatten(),
                classification_prediction.flatten()
            ]),
            index=prediction_dates,
            columns=["预测方向", "预测概率"])
        # Use the original output_dir (Drive path) for CSV
        classification_path = os.path.join(output_dir, f"{stock_code}_{today}_classification_predictions.csv")
        classification_df.to_csv(classification_path)
        print(f"✅ 分类预测CSV已保存至 Google Drive：{classification_path}")
    except Exception as e:
        print(f"❌ 分类CSV保存失败：{stock_code} → {e}")

# Modify the run_backtest function to return predicted and actual values
def run_backtest(stock_code, stock_data, model_builder_func, window_size, pred_size, drive_model_dir, epochs=20, backtest_days=None):
    """
    Runs a rolling window backtest for a given stock.

    Args:
        stock_code (str): The stock code.
        stock_data (dict): Dictionary containing df, df_scaled, etc. for the stock.
        model_builder_func (function): Function to build a new model.
        window_size (int): The size of the look-back window for training.
        pred_size (int): The size of the prediction window.
        drive_model_dir (str): Directory in Google Drive to save models.
        epochs (int): Number of training epochs per step.
        backtest_days (int, optional): Limits the backtest to the last N steps. Defaults to None (full backtest).

    Returns:
        dict: Dictionary containing collected predicted and actual values and dates.
              Returns None if backtest cannot be run.
    """
    df = stock_data["df"]
    df_scaled = stock_data["df_scaled"]
    target_regression_data = stock_data["target_regression_data"]
    target_classification_data = stock_data["target_classification_data"]
    scaler = stock_data["scaler"]
    actual_regression_target_names = stock_data["actual_regression_target_names"]

    # Determine the start and end indices for backtesting
    # We need enough data for the first training window and the first prediction window
    if len(df_scaled) < window_size + pred_size:
        print(f"⚠️ 数据不足以进行回测 ({window_size} + {pred_size})，跳过回测：{stock_code}")
        return None

    # The backtest starts from the first date where we can make a prediction
    # This is after the initial training window (window_size) has passed.
    # The loop will iterate through each day that will be the start of a prediction window.
    backtest_start_index = window_size
    backtest_end_index = len(df_scaled) - pred_size # The last possible start date for a full prediction window

    # Apply backtest_days limit if specified
    if backtest_days is not None and backtest_days > 0:
        # Calculate the starting index for the limited backtest
        limited_backtest_start_index = max(backtest_start_index, backtest_end_index - backtest_days + 1)
        print(f"Limiting backtest to the last {backtest_days} steps.")
        print(f"Original backtest range: {df.index[backtest_start_index].strftime('%Y-%m-%d')} to {df.index[backtest_end_index].strftime('%Y-%m-%d')}")
        backtest_start_index = limited_backtest_start_index


    print(f"🏃‍♂️ 开始对股票 {stock_code} 进行滚动回测...")
    print(f"回测数据范围: {df.index[backtest_start_index].strftime('%Y-%m-%d')} to {df.index[backtest_end_index].strftime('%Y-%m-%d')}")


    # Initialize lists to store results for this stock and combination
    stock_predicted_regression = []
    stock_actual_regression = []
    stock_predicted_classification = []
    stock_actual_classification = []
    stock_dates = []


    # Iterate through each date that will be the start of the prediction window
    for current_pred_start_index in range(backtest_start_index, backtest_end_index + 1):
        # Define the training window (ends the day before the prediction starts)
        train_start_index = current_pred_start_index - window_size
        train_end_index = current_pred_start_index - 1
        train_data_scaled_window = df_scaled[train_start_index : train_end_index + 1]
        train_target_regression_window = target_regression_data[train_start_index : train_end_index + 1]
        train_target_classification_window = target_classification_data[train_start_index : train_end_index + 1]


        # Define the prediction window indices for collecting actuals
        pred_start_index = current_pred_start_index
        pred_end_index = current_pred_start_index + pred_size
        # Ensure prediction window does not go beyond available data for actual values
        actual_pred_end_index = min(pred_end_index, len(df_scaled))
        prediction_indices_for_actuals = range(pred_start_index, actual_pred_end_index)


        # Skip if the prediction window is empty or too short
        # Check if there are enough actual data points for the prediction window
        if not prediction_indices_for_actuals or len(prediction_indices_for_actuals) < pred_size:
             # This check might not be necessary with the loop range calculation, but added for safety
             print(f"⚠️ 实际预测窗口太短，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             continue

        # Prepare the single training sample for this step
        # The training sample is the sequence of 'window_size' data points
        # and the targets are the next 'pred_size' data points.
        # Ensure the targets for training are from the correct indices (the 'pred_size' days *after* the training window)
        X_train_sample = df_scaled[train_start_index : train_end_index + 1]
        y_train_regression_sample = target_regression_data[pred_start_index : pred_end_index]
        y_train_classification_sample = target_classification_data[pred_start_index : pred_end_index]

        # Reshape samples to match model input/output expectations (batch_size, window_size, features)
        X_train_sample = np.expand_dims(X_train_sample, axis=0)
        y_train_regression_sample = np.expand_dims(y_train_regression_sample, axis=0)
        y_train_classification_sample = np.expand_dims(y_train_classification_sample, axis=0)
        # Ensure classification target has the last dimension as 1
        if y_train_classification_sample.ndim == 2:
             y_train_classification_sample = np.expand_dims(y_train_classification_sample, axis=-1)


        # Validate shapes before training
        expected_X_shape = (1, window_size, df_scaled.shape[1])
        expected_y_reg_shape = (1, pred_size, target_regression_data.shape[1])
        expected_y_clf_shape = (1, pred_size, 1)

        if X_train_sample.shape != expected_X_shape or y_train_regression_sample.shape != expected_y_reg_shape or y_train_classification_sample.shape != expected_y_clf_shape:
             print(f"❌ 训练样本形状不匹配，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             print(f"Expected X shape: {expected_X_shape}, Actual X shape: {X_train_sample.shape}")
             print(f"Expected y_reg shape: {expected_y_reg_shape}, Actual y_reg shape: {y_train_regression_sample.shape}")
             print(f"Expected y_clf shape: {expected_y_clf_shape}, Actual y_clf shape: {y_train_classification_sample.shape}")
             continue


        # Model training logic
        model_path_step = os.path.join(drive_model_dir, f"{stock_code}_transformer_model_step_{df.index[current_pred_start_index].strftime('%Y%m%d')}.h5")

        # Train the model on the single sample for this step
        print(f"🔄 正在训练模型，回测步预测开始日期: {df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
        try:
            # Use the model_builder_func to get a fresh model for this step
            model = build_transformer_model((window_size, df_scaled.shape[1]), target_regression_data.shape[1], pred_size)

            # Define callbacks (optional for single step training but good practice)
            # checkpoint_cb = ModelCheckpoint(model_path_step, save_best_only=False, save_freq='epoch', verbose=0)
            # tqdm_callback = TqdmCallback(verbose=0) # Use verbose=0 to avoid excessive output per step

            model.fit(X_train_sample,
                      {"regression_output": y_train_regression_sample, "classification_output": y_train_classification_sample},
                      epochs=epochs,
                      batch_size=1, # Train on a single sample
                      verbose=0, # Disable fit's verbose, use TqdmCallback if enabled
                      # callbacks=[tqdm_callback] # Only use tqdm_callback if desired
                      )

            # Save the model after training for this step
            try:
                model.save(model_path_step)
                print(f"✅ 模型已保存至：{model_path_step}")
            except Exception as save_error:
                print(f"❌ 模型保存失败：{model_path_step} → {save_error}")

        except Exception as e:
             print(f"❌ 模型训练失败，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')} → {e}")
             continue


        # Prepare data for prediction (the single sample to predict from)
        # This is the sequence of 'window_size' data points ending the day before the prediction starts.
        predict_data_scaled = df_scaled[current_pred_start_index - window_size : current_pred_start_index]

        # Ensure predict_data_scaled has the correct shape (window_size, num_features)
        if predict_data_scaled.shape[0] != window_size or predict_data_scaled.shape[1] != df_scaled.shape[1]:
             print(f"❌ 预测输入数据维度错误，跳过预测：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             continue

        # Make prediction
        try:
            predictions = model.predict(np.expand_dims(predict_data_scaled, axis=0), verbose=0) # Predict on a single sample
            regression_prediction = predictions[0][0] # Shape: (pred_size, num_regression_targets)
            classification_prediction = predictions[1][0] # Shape: (pred_size, 1)
        except Exception as e:
            print(f"❌ 模型预测失败，跳过当前回测试步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')} → {e}")
            continue

        # Store actual values and dates for the prediction window
        # Ensure we only collect actuals for the dates where prediction was made and actual data exists
        actual_regression_window = target_regression_data[pred_start_index : actual_pred_end_index]
        actual_classification_window = target_classification_data[pred_start_index : actual_pred_end_index]
        prediction_dates = df.index[pred_start_index : actual_pred_end_index]

        # Store the results for this step
        stock_predicted_regression.extend(regression_prediction[:len(prediction_indices_for_actuals)])
        stock_actual_regression.extend(actual_regression_window)
        stock_predicted_classification.extend(classification_prediction[:len(prediction_indices_for_actuals)].flatten())
        stock_actual_classification.extend(actual_classification_window.flatten())
        stock_dates.extend(prediction_dates.tolist())


    # Consolidate results for this stock and combination into a DataFrame
    if not stock_dates:
        print(f"⚠️ 未生成任何回测结果，跳过结果保存：{stock_code}")
        return None

    # Create DataFrame from collected data
    # Need to ensure consistent structure for regression targets when creating DataFrame
    # Each element in stock_predicted_regression and stock_actual_regression is a list of size num_regression_targets
    # Let's convert these lists of lists/arrays into a single 2D numpy array
    try:
        predicted_regression_array = np.array(stock_predicted_regression)
        actual_regression_array = np.array(stock_actual_regression)
        predicted_classification_array = np.array(stock_predicted_classification)
        actual_classification_array = np.array(stock_actual_classification)

    except Exception as e:
        print(f"❌ 汇总回测结果时数据转换失败：{stock_code} → {e}")
        return None

    # Create column names for regression targets based on actual_regression_target_names
    regression_pred_cols = [f"Predicted_{name}" for name in actual_regression_target_names]
    regression_actual_cols = [f"Actual_{name}" for name in actual_regression_target_names]

    # Create DataFrame with all collected results
    results_data = {}
    results_data["Date"] = stock_dates
    # Add regression columns
    for i, col_name in enumerate(regression_pred_cols):
        results_data[col_name] = predicted_regression_array[:, i]
    for i, col_name in enumerate(regression_actual_cols):
        results_data[col_name] = actual_regression_array[:, i]

    # Add classification columns
    results_data["Predicted_Classification"] = predicted_classification_array
    results_data["Actual_Classification"] = actual_classification_array


    results_df = pd.DataFrame(results_data)

    results_df["Date"] = pd.to_datetime(results_df["Date"])
    results_df.set_index("Date", inplace=True)
    results_df.sort_index(inplace=True)

    # Handle potential duplicate dates by keeping the last prediction for each date
    results_df = results_df[~results_df.index.duplicated(keep='last')].copy()


    print(f"✅ 滚动回测对股票 {stock_code} 完成.")

    return results_df


# --- Main Backtesting Loop ---

# Define different window sizes and prediction sizes for experiments
window_sizes_to_try = [30, 45]
pred_sizes_to_try = [3, 5, 8]

# Control whether to run only the first stock for debugging
debug_mode = False # Set to True to run only the first stock for faster debugging

# Set the number of backtest steps to run (e.g., last 10 days)
backtest_days_limit = 10 # Set this to the desired number of backtest steps

# Load complete data for each stock outside the main loop
all_stocks_data = {}
# Use the stock_codes defined at the beginning of the cell
stock_codes = ['601138','002269'] # Ensure stock_codes is defined here
start_date = "2024-09-30" # Ensure start_date is defined
end_date = "2025-09-30"   # Ensure end_date is defined


for code in stock_codes:
    print(f"加载完整数据用于回测: {code}")
    # load_daily_and_fundflow now loads the full range before slicing
    df, df_scaled, target_regression_data, target_classification_data, scaler, actual_regression_target_names = load_daily_and_fundflow(code, start_date, end_date)

    # Check if target_regression_data and target_classification_data are not None and not empty
    if (df is not None and not df.empty and
        target_regression_data is not None and target_regression_data.size > 0 and
        target_classification_data is not None and target_classification_data.size > 0):
        all_stocks_data[code] = {
            "df": df,
            "df_scaled": df_scaled,
            "target_regression_data": target_regression_data,
            "target_classification_data": target_classification_data,
            "scaler": scaler,
            "actual_regression_target_names": actual_regression_target_names
        }
    else:
        print(f"跳过股票 {code} 的回测数据加载，数据为空或目标数据缺失。")


# Reset results to store backtest results
results = {}

# Define the base directory for saving results and models in Google Drive
# Make sure drive_output_base_dir is defined (assuming previous cell was run)
# If not, define it here:
if 'drive_output_base_dir' not in globals():
     drive_output_base_dir = "/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results"
     os.makedirs(drive_output_base_dir, exist_ok=True)
     print(f"Base output directory in Google Drive: {drive_output_base_dir}")


# Loop through stock codes first
stocks_to_process = stock_codes
if debug_mode:
    stocks_to_process = [stock_codes[0]] # Debug mode: process only the first stock


for code in stocks_to_process:
    if code not in all_stocks_data:
        print(f"跳过股票 {code}，回测数据未加载。")
        continue

    stock_data = all_stocks_data[code]
    print(f"\n--- 对股票 {code} 进行窗口组合测试 ---")

    # Loop through different window size combinations
    for current_window_size in window_sizes_to_try:
        for current_pred_size in pred_sizes_to_try:
            print(f"\n--- 测试组合: 回溯窗口={current_window_size}, 预测窗口={current_pred_size} ---")

            # Define the directory to save models for this specific window/pred size combination within Drive
            drive_model_sub_dir = os.path.join(drive_output_base_dir, f"window_{current_window_size}_pred_{current_pred_size}", "models")
            os.makedirs(drive_model_sub_dir, exist_ok=True)

            # Run the backtest for the current stock and window combination, applying the backtest_days_limit
            backtest_results_df = run_backtest(
                stock_code=code,
                stock_data=stock_data,
                model_builder_func=build_transformer_model, # Pass the function
                window_size=current_window_size,
                pred_size=current_pred_size,
                drive_model_dir=drive_model_sub_dir, # Pass the Drive model directory
                epochs=1, # Reduced epochs significantly for testing this iteration
                backtest_days=backtest_days_limit # Pass the backtest days limit
            )

            if backtest_results_df is not None:
                # Store results for the current combination
                combination_key = f"{code}_window_{current_window_size}_pred_{current_pred_size}"
                results[combination_key] = backtest_results_df

                # Save raw backtest results to CSV in Google Drive
                drive_results_sub_dir = os.path.join(drive_output_base_dir, f"window_{current_window_size}_pred_{current_pred_size}", "results")
                os.makedirs(drive_results_sub_dir, exist_ok=True)
                results_path_drive = os.path.join(drive_results_sub_dir, f"{code}_backtest_results.csv")
                backtest_results_df.to_csv(results_path_drive)
                print(f"✅ 回测结果CSV已保存至 Google Drive：{results_path_drive}")

                # Note: Plotting and saving charts within the rolling window backtest
                # is generally not done at each step. The charts and performance metrics
                # should be generated *after* the backtest from the aggregated results_df.
                # The predict_and_plot_targets function can be used separately to visualize
                # the *final* prediction after the backtest is complete, or for debugging
                # individual steps if adapted.

# After all backtests are run, you can analyze the 'results' dictionary to calculate
# overall performance metrics for each window size combination and each stock.

print("\n--- 所有回测任务完成 ---")
# Further steps would involve analyzing the 'results' dictionary, calculating metrics (e.g., accuracy, MAE, Sharpe Ratio), and visualizing.
# You would iterate through the 'results' dictionary and process each backtest_results_df.
# Example: Calculate classification accuracy for each combination
# for key, backtest_df in results.items():
#     actual_clf = backtest_df["Actual_Classification"].values
#     predicted_clf = (backtest_df["Predicted_Classification"].values > 0.5).astype(int)
#     accuracy = accuracy_score(actual_clf, predicted_clf)
#     print(f"Combination {key} Classification Accuracy: {accuracy:.4f}")

加载完整数据用于回测: 601138
💰 资金流数据合并成功：601138
加载完整数据用于回测: 002269
💰 资金流数据合并成功：002269

--- 对股票 601138 进行窗口组合测试 ---

--- 测试组合: 回溯窗口=30, 预测窗口=3 ---
Limiting backtest to the last 10 steps.
Original backtest range: 2024-12-05 to 2025-09-26
🏃‍♂️ 开始对股票 601138 进行滚动回测...
回测数据范围: 2025-09-15 to 2025-09-26
🔄 正在训练模型，回测步预测开始日期: 2025-09-15


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_30_pred_3/models/601138_transformer_model_step_20250915.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-16


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_30_pred_3/models/601138_transformer_model_step_20250916.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-17


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_30_pred_3/models/601138_transformer_model_step_20250917.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-18


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_30_pred_3/models/601138_transformer_model_step_20250918.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-19


...

✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_45_pred_8/models/002269_transformer_model_step_20250915.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-16


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_45_pred_8/models/002269_transformer_model_step_20250916.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-17


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_45_pred_8/models/002269_transformer_model_step_20250917.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-18


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_45_pred_8/models/002269_transformer_model_step_20250918.h5
🔄 正在训练模型，回测步预测开始日期: 2025-09-19


✅ 模型已保存至：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_45_pred_8/models/002269_transformer_model_step_20250919.h5
✅ 滚动回测对股票 002269 完成.
✅ 回测结果CSV已保存至 Google Drive：/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results/window_45_pred_8/results/002269_backtest_results.csv

--- 所有回测任务完成 ---


In [ ]:
# 作为世界顶级的量化专家，我们深知回溯窗口 (window_size) 和预测窗口 (pred_size) 对时间序列模型性能的关键影响。
# 为了找到最优的窗口组合，我们将通过实验来探索不同的参数设置。

# 定义不同的窗口大小组合进行实验
# 这里的组合仅为示例，你可以根据你的数据和研究目标调整这些值。
window_sizes_to_try = [15, 30, 45]
pred_sizes_to_try = [1, 3, 5]

# 控制是否只训练一支股票进行调试
debug_mode = False # 设置为 True 只运行第一支股票，方便快速调试

# 存储不同组合的模型性能结果，方便后续分析对比
results = {}

# 循环遍历不同的窗口大小组合
for current_window_size in window_sizes_to_try:
    for current_pred_size in pred_sizes_to_try:
        print(f"\n--- 正在测试组合: 回溯窗口={current_window_size}, 预测窗口={current_pred_size} ---")

        # 存储当前组合的数据和模型
        current_combination_data = {}
        trained_models = {}

        # 遍历每支股票，加载数据、处理特征、创建序列并训练模型
        stocks_to_process = stock_codes
        if debug_mode:
            stocks_to_process = [stock_codes[0]] # 调试模式下只处理第一支股票

        for code in stocks_to_process:
            print(f"处理股票: {code}")
            # 加载日线和资金流数据
            df, df_scaled, target_regression_data, target_classification_data, scaler, actual_regression_target_names = load_daily_and_fundflow(code, start_date, end_date)

            if df is None or df.empty:
                print(f"跳过股票 {code}，数据加载或处理失败。")
                continue

            # 使用当前的窗口大小和预测大小创建序列
            X, y_regression, y_classification = create_sequences(df_scaled, target_regression_data, target_classification_data, window_size=current_window_size, pred_size=current_pred_size)

            # 检查序列是否成功创建且长度符合要求
            if X.shape[0] == 0:
                print(f"跳过股票 {code}，数据序列创建失败或数据不足。")
                continue
            if X.shape[1:] != (current_window_size, df_scaled.shape[1]):
                 print(f"❌ 股票 {code} 的输入序列形状不匹配，跳过训练。预期: {(current_window_size, df_scaled.shape[1])}, 实际: {X.shape[1:]}")
                 continue
            if y_regression.shape[1:] != (current_pred_size, target_regression_data.shape[1]):
                 print(f"❌ 股票 {code} 的回归目标序列形状不匹配，跳过训练。预期: {(current_pred_size, target_regression_data.shape[1])}, 实际: {y_regression.shape[1:]}")
                 continue
            if y_classification.shape[1:] != (current_pred_size, 1): # 分类目标通常是单列
                  print(f"❌ 股票 {code} 的分类目标序列形状不匹配，跳过训练。预期: {(current_pred_size, 1)}, 实际: {y_classification.shape[1:]}")
                  continue


            # 打印序列形状，方便检查
            print(f"股票 {code} 的序列形状: X={X.shape}, y_regression={y_regression.shape}, y_classification={y_classification.shape}")

            # 保存处理后的数据和 scaler，用于后续预测时的反标准化
            current_combination_data[code] = {
                "df": df,
                "df_scaled": df_scaled,
                "scaler": scaler,
                "actual_regression_target_names": actual_regression_target_names
            }

            # 定义模型保存路径
            model_dir = f"./models/window_{current_window_size}_pred_{current_pred_size}"
            os.makedirs(model_dir, exist_ok=True)
            model_path = os.path.join(model_dir, f"{code}_transformer_model.h5")

            # 训练模型
            print(f"训练股票 {code} 的模型...")
            # 注意：这里为了快速演示，epochs 设置得比较小，实际应用中可能需要更多 epochs
            # 使用 try-except 块捕获 KeyboardInterrupt，以便在中断时也能保存部分结果
            try:
                trained_model = train_transformer_model(X, y_regression, y_classification, X.shape[1:], model_path, epochs=10)
                trained_models[code] = trained_model
                print(f"股票 {code} 模型训练完成并保存到 {model_path}")
            except KeyboardInterrupt:
                print(f"\n🛑 训练股票 {code} 时被用户中断。")
                # 可以在这里添加保存当前进度的逻辑，如果需要的话
                # 注意：ModelCheckpoint 已经在回调中处理了模型的保存
                break # 中断当前股票的训练，并跳出股票循环
            except Exception as e:
                print(f"❌ 训练股票 {code} 时发生错误：{e}")
                continue # 继续处理下一支股票


        # 在所有股票上进行预测并保存图表和结果
        output_dir = f"./charts/window_{current_window_size}_pred_{current_pred_size}"
        os.makedirs(output_dir, exist_ok=True)

        combination_results = {}
        for code in stocks_to_process: # 在调试模式下只预测调试的股票
             if code in trained_models and code in current_combination_data:
                print(f"对股票 {code} 进行预测...")
                data_info = current_combination_data[code]
                model = trained_models[code]
                # 确保预测时使用的窗口大小与训练时一致
                predict_and_plot_targets(model, data_info["df"], data_info["df_scaled"], data_info["scaler"], current_window_size, code, data_info["actual_regression_target_names"], output_dir=output_dir)

                # 这里可以添加代码来收集和存储模型的评估指标，以便后续比较
                # 例如：从模型训练历史中提取损失和指标，或在独立的测试集上进行评估

        # 可以将每个组合的整体性能总结存储在 results 字典中
        # results[f"window_{current_window_size}_pred_{current_pred_size}"] = overall_performance_metrics

# 循环结束后，你可以分析 results 字典来比较不同窗口组合的性能
print("\n--- 所有窗口组合测试完成 ---")
# 可以在这里添加代码来打印或可视化 results 字典中的性能对比


--- 正在测试组合: 回溯窗口=15, 预测窗口=1 ---
处理股票: 601138
💰 资金流数据合并成功：601138
股票 601138 的序列形状: X=(217, 15, 22), y_regression=(217, 1, 3), y_classification=(217, 1, 1)
训练股票 601138 的模型...
✅ 检测到已保存模型，正在加载：./models/window_15_pred_1/601138_transformer_model.h5
❌ 训练股票 601138 时发生错误：too many positional arguments
处理股票: 002269
💰 资金流数据合并成功：002269
股票 002269 的序列形状: X=(217, 15, 22), y_regression=(217, 1, 3), y_classification=(217, 1, 1)
训练股票 002269 的模型...
✅ 检测到已保存模型，正在加载：./models/window_15_pred_1/002269_transformer_model.h5
❌ 训练股票 002269 时发生错误：too many positional arguments

--- 正在测试组合: 回溯窗口=15, 预测窗口=3 ---
处理股票: 601138
💰 资金流数据合并成功：601138
股票 601138 的序列形状: X=(215, 15, 22), y_regression=(215, 3, 3), y_classification=(215, 3, 1)
训练股票 601138 的模型...
✅ 检测到已保存模型，正在加载：./models/window_15_pred_3/601138_transformer_model.h5
❌ 训练股票 601138 时发生错误：too many positional arguments
处理股票: 002269
💰 资金流数据合并成功：002269
股票 002269 的序列形状: X=(215, 15, 22), y_regression=(215, 3, 3), y_classification=(215, 3, 1)
训练股票 002269 的模型...
✅ 检测到已保存模型，正在加

In [ ]:
# Modify the load_daily_and_fundflow function to remove date slicing
def load_daily_and_fundflow(code, start_date, end_date):
    prefix = code[-6:]
    market_prefix = "sz" if code.startswith("0") else "sh"
    daily_path = f"/content/drive/MyDrive/Colab Notebooks/ant_data/kline/{prefix}_daily.csv"
    fundflow_path = f"/content/drive/MyDrive/Colab Notebooks/ant_data/jiqi_do/{market_prefix}{code}.csv"

    if not os.path.exists(daily_path):
        print(f"⚠️ 缺失日线数据文件：{daily_path}")
        return [None] * 6

    try:
        df = pd.read_csv(daily_path)
        df = standardize_columns(df.copy())
        if "Date" not in df.columns:
            print(f"❌ 'Date'列缺失：{daily_path}")
            return [None] * 6

        df["Date"] = pd.to_datetime(df["Date"])
        df.set_index("Date", inplace=True)
        df.sort_index(inplace=True)
        df = df[~df.index.duplicated(keep='last')]
        # Removed date slicing here: df = df.loc[pd.to_datetime(start_date):pd.to_datetime(end_date)].copy()

        if os.path.exists(fundflow_path):
            df_fund = pd.read_csv(fundflow_path)
            fundflow_column_mapping = {
                "opendate": "Date",
                "netamount": "MainNetInflow",
                "ratioamount": "MainInflowRatio"
            }
            df_fund.rename(columns=fundflow_column_mapping, inplace=True)
            df_fund.drop(columns=[col for col in ["turnover", "trade", "changeratio"] if col in df_fund.columns], inplace=True)

            if "Date" in df_fund.columns:
                df_fund["Date"] = pd.to_datetime(df_fund["Date"])
                df_fund.set_index("Date", inplace=True)
                df = df.join(df_fund, how="left")
                print(f"💰 资金流数据合并成功：{code}")
            else:
                print(f"❌ 资金流数据缺少 'Date' 列：{code}")
        else:
            print(f"⚠️ 缺失资金流文件：{fundflow_path}")

        for col in ["MainNetInflow", "MainInflowRatio"]:
            if col not in df.columns:
                df[col] = 0
                print(f"⚠️ 初始化缺失资金流字段：{col}")

        # Apply date slicing after feature processing to ensure all data is available for indicators
        df = df.loc[pd.to_datetime(start_date):pd.to_datetime(end_date)].copy()
        if df.empty:
            print(f"⚠️ 切片后数据为空：{code}")
            return [None] * 6

        return process_features(df)

    except Exception as e:
        print(f"❌ 数据处理失败：{code} → {e}")
        return [None] * 6

# Load complete data for each stock outside the main loop
all_stocks_data = {}
for code in stock_codes:
    print(f"加载完整数据用于回测: {code}")
    # load_daily_and_fundflow now loads the full range before slicing
    df, df_scaled, target_regression_data, target_classification_data, scaler, actual_regression_target_names = load_daily_and_fundflow(code, start_date, end_date)

    if df is not None and not df.empty:
        all_stocks_data[code] = {
            "df": df,
            "df_scaled": df_scaled,
            "target_regression_data": target_regression_data,
            "target_classification_data": target_classification_data,
            "scaler": scaler,
            "actual_regression_target_names": actual_regression_target_names
        }
    else:
        print(f"跳过股票 {code} 的回测数据加载，数据为空。")

# The rest of the code for training and backtesting will use data from all_stocks_data
# Note: The subsequent training loop in the original notebook will need to be adapted
# to use this pre-loaded data and implement the rolling window backtesting logic.

加载完整数据用于回测: 601138
💰 资金流数据合并成功：601138
加载完整数据用于回测: 002269
💰 资金流数据合并成功：002269


In [ ]:
def run_backtest(stock_code, stock_data, model_builder_func, window_size, pred_size, model_dir="./models", output_dir="./charts", epochs=20):
    """
    Runs a rolling window backtest for a given stock.

    Args:
        stock_code (str): The stock code.
        stock_data (dict): Dictionary containing df, df_scaled, etc. for the stock.
        model_builder_func (function): Function to build a new model.
        window_size (int): The size of the look-back window for training.
        pred_size (int): The size of the prediction window.
        model_dir (str): Directory to save/load models.
        output_dir (str): Directory to save charts and results.
        epochs (int): Number of training epochs per step.

    Returns:
        dict: Dictionary containing collected predicted and actual values and dates.
    """
    df = stock_data["df"]
    df_scaled = stock_data["df_scaled"]
    target_regression_data = stock_data["target_regression_data"]
    target_classification_data = stock_data["target_classification_data"]
    scaler = stock_data["scaler"]
    actual_regression_target_names = stock_data["actual_regression_target_names"]

    # Determine the start and end indices for backtesting
    # We need enough data for the first training window and the first prediction window
    if len(df_scaled) < window_size + pred_size:
        print(f"⚠️ 数据不足以进行回测 ({window_size} + {pred_size})，跳过回测：{stock_code}")
        return None

    # The backtest starts from the first date where we can make a prediction
    # This is after the initial training window (window_size) has passed.
    # The loop will iterate through each day that will be the start of a prediction window.
    backtest_start_index = window_size
    backtest_end_index = len(df_scaled) - pred_size # The last possible start date for a full prediction window

    print(f"🏃‍♂️ 开始对股票 {stock_code} 进行滚动回测...")
    print(f"回测数据范围: {df.index[backtest_start_index]} to {df.index[backtest_end_index]}")

    all_predicted_regression = []
    all_actual_regression = []
    all_predicted_classification = []
    all_actual_classification = []
    all_dates = []

    # Iterate through each date that will be the start of the prediction window
    for current_pred_start_index in range(backtest_start_index, backtest_end_index + 1):
        # Define the training window (ends the day before the prediction starts)
        train_start_index = current_pred_start_index - window_size
        train_end_index = current_pred_start_index - 1
        train_data_scaled_window = df_scaled[train_start_index : train_end_index + 1]
        train_target_regression_window = target_regression_data[train_start_index : train_end_index + 1]
        train_target_classification_window = target_classification_data[train_start_index : train_end_index + 1]

        # Define the prediction window
        pred_start_index = current_pred_start_index
        pred_end_index = current_pred_start_index + pred_size
        # Ensure prediction window does not go beyond available data for actual values
        actual_pred_end_index = min(pred_end_index, len(df_scaled))
        prediction_indices = range(pred_start_index, actual_pred_end_index)

        # Skip if the prediction window is empty or too short
        if not prediction_indices or len(prediction_indices) < pred_size:
             # This check might not be necessary with the loop range calculation, but added for safety
             print(f"⚠️ 预测窗口太短，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             continue

        # Prepare the single training sample for this step
        # The training sample is the sequence of 'window_size' data points
        # and the targets are the next 'pred_size' data points.
        X_train_sample = df_scaled[train_start_index : train_end_index + 1]
        y_train_regression_sample = target_regression_data[pred_start_index : pred_end_index]
        y_train_classification_sample = target_classification_data[pred_start_index : pred_end_index]

        # Reshape samples to match model input/output expectations (batch_size, window_size, features)
        X_train_sample = np.expand_dims(X_train_sample, axis=0)
        y_train_regression_sample = np.expand_dims(y_train_regression_sample, axis=0)
        y_train_classification_sample = np.expand_dims(y_train_classification_sample, axis=0)
        # Ensure classification target has the last dimension as 1
        if y_train_classification_sample.ndim == 2:
             y_train_classification_sample = np.expand_dims(y_train_classification_sample, axis=-1)


        if X_train_sample.shape[1] != window_size or y_train_regression_sample.shape[1] != pred_size or y_train_classification_sample.shape[1] != pred_size:
             print(f"❌ 训练样本形状不匹配，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             print(f"Expected X shape: (1, {window_size}, {df_scaled.shape[1]}), Actual X shape: {X_train_sample.shape}")
             print(f"Expected y_reg shape: (1, {pred_size}, {target_regression_data.shape[1]}), Actual y_reg shape: {y_train_regression_sample.shape}")
             print(f"Expected y_clf shape: (1, {pred_size}, 1), Actual y_clf shape: {y_train_classification_sample.shape}")
             continue


        # Model training/loading logic
        model_sub_dir = f"window_{window_size}_pred_{pred_size}"
        model_path = os.path.join(model_dir, model_sub_dir, f"{stock_code}_transformer_model.h5")

        # Train the model on the single sample for this step
        print(f"🔄 正在训练模型，回测步预测开始日期: {df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
        try:
            # Use the model_builder_func to get a fresh model or load an existing one
            # For simplicity in this step, let's train from scratch at each step.
            # A more advanced backtest would load and potentially update.
            model = build_transformer_model((window_size, df_scaled.shape[1]), target_regression_data.shape[1], pred_size)
            model.fit(X_train_sample,
                      {"regression_output": y_train_regression_sample, "classification_output": y_train_classification_sample},
                      epochs=epochs,
                      batch_size=1, # Train on a single sample
                      verbose=0)
            # Note: Saving the model at every step might be too slow/storage intensive.
            # Consider saving only periodically or at the end.
            # model.save(model_path) # Optional: Save model

        except Exception as e:
             print(f"❌ 模型训练失败，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')} → {e}")
             continue


        # Prepare data for prediction (the single sample to predict from)
        # This is the sequence of 'window_size' data points ending the day before the prediction starts.
        predict_data_scaled = df_scaled[current_pred_start_index - window_size : current_pred_start_index]

        # Ensure predict_data_scaled has the correct shape (window_size, num_features)
        if predict_data_scaled.shape[0] != window_size or predict_data_scaled.shape[1] != df_scaled.shape[1]:
             print(f"❌ 预测输入数据维度错误，跳过预测：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             continue

        # Make prediction
        try:
            predictions = model.predict(np.expand_dims(predict_data_scaled, axis=0), verbose=0) # Predict on a single sample
            regression_prediction = predictions[0][0] # Shape: (pred_size, num_regression_targets)
            classification_prediction = predictions[1][0] # Shape: (pred_size, 1)
        except Exception as e:
            print(f"❌ 模型预测失败，跳过当前回测试步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')} → {e}")
            continue

        # Store actual values and dates for the prediction window
        actual_regression_window = target_regression_data[pred_start_index : actual_pred_end_index]
        actual_classification_window = target_classification_data[pred_start_index : actual_pred_end_index]
        prediction_dates = df.index[pred_start_index : actual_pred_end_index]

        # Store the results, only for the dates where actual data is available
        all_predicted_regression.extend(regression_prediction[:len(prediction_indices)])
        all_actual_regression.extend(actual_regression_window)
        # Extend with individual elements from the prediction window
        all_predicted_classification.extend(classification_prediction[:len(prediction_indices)].flatten())
        all_actual_classification.extend(actual_classification_window.flatten())
        all_dates.extend(prediction_dates.tolist())


    # Concatenate results and handle potential overlaps (take the prediction closest to the date)
    if not all_dates:
        print(f"⚠️ 未生成任何回测结果，跳过结果保存：{stock_code}")
        return None

    # Ensure actual and predicted data have consistent shapes before creating DataFrame
    # They should be lists of arrays/lists of size num_targets for regression and 1 for classification per date
    # Let's flatten the lists of arrays for easier DataFrame creation
    flat_predicted_regression = [item for sublist in all_predicted_regression for item in sublist]
    flat_actual_regression = [item for sublist in all_actual_regression for item in sublist]
    # Directly convert to numpy arrays and reshape
    flat_predicted_classification = np.array(all_predicted_classification).reshape(-1, 1)
    flat_actual_classification = np.array(all_actual_classification).reshape(-1, 1)


    results_df = pd.DataFrame({
        "Date": all_dates,
        "Predicted_Regression": list(flat_predicted_regression), # Store as list of arrays/lists
        "Actual_Regression": list(flat_actual_regression),
        "Predicted_Classification": list(flat_predicted_classification),
        "Actual_Classification": list(flat_actual_classification)
    })

    results_df["Date"] = pd.to_datetime(results_df["Date"])
    results_df.set_index("Date", inplace=True)
    results_df.sort_index(inplace=True)

    # Handle potential duplicate dates by keeping the last prediction for each date
    # This is a simplification; a proper backtest would handle multi-step predictions
    # and their validity over time more carefully.
    results_df = results_df[~results_df.index.duplicated(keep='last')].copy()


    print(f"✅ 滚动回测对股票 {stock_code} 完成.")

    return results_df

# Now, adapt the main loop to use run_backtest
# We will loop through stock codes and then call run_backtest for each combination of window/pred size
results = {} # Reset results to store backtest results

# Loop through stock codes first
for code in stock_codes:
    if code not in all_stocks_data:
        print(f"跳过股票 {code}，回测数据未加载。")
        continue

    stock_data = all_stocks_data[code]
    print(f"\n--- 对股票 {code} 进行窗口组合测试 ---")

    # Loop through different window size combinations
    for current_window_size in window_sizes_to_try:
        for current_pred_size in pred_sizes_to_try:
            print(f"\n--- 测试组合: 回溯窗口={current_window_size}, 预测窗口={current_pred_size} ---")

            # Run the backtest for the current stock and window combination
            backtest_results_df = run_backtest(
                stock_code=code,
                stock_data=stock_data,
                model_builder_func=build_transformer_model, # Pass the function
                window_size=current_window_size,
                pred_size=current_pred_size,
                model_dir="./models",
                output_dir=f"./charts/window_{current_window_size}_pred_{current_pred_size}",
                epochs=1 # Reduced epochs significantly for testing this iteration
            )

            if backtest_results_df is not None:
                # Store results for the current combination
                combination_key = f"{code}_window_{current_window_size}_pred_{current_pred_size}"
                results[combination_key] = backtest_results_df

                # Optional: Save raw backtest results to CSV
                output_dir_comb = f"./charts/window_{current_window_size}_pred_{current_pred_size}"
                os.makedirs(output_dir_comb, exist_ok=True)
                results_path = os.path.join(output_dir_comb, f"{code}_backtest_results.csv")
                backtest_results_df.to_csv(results_path)
                print(f"✅ 回测结果CSV已保存：{results_path}")

# After all backtests are run, you can analyze the 'results' dictionary
print("\n--- 所有回测任务完成 ---")
# Further steps would involve analyzing the 'results' dictionary, calculating metrics, and visualizing.


--- 对股票 601138 进行窗口组合测试 ---

--- 测试组合: 回溯窗口=15, 预测窗口=1 ---
🏃‍♂️ 开始对股票 601138 进行滚动回测...
回测数据范围: 2024-11-14 00:00:00 to 2025-09-30 00:00:00
🔄 正在训练模型，回测步预测开始日期: 2024-11-14
🔄 正在训练模型，回测步预测开始日期: 2024-11-15
🔄 正在训练模型，回测步预测开始日期: 2024-11-18
🔄 正在训练模型，回测步预测开始日期: 2024-11-19
🔄 正在训练模型，回测步预测开始日期: 2024-11-20
🔄 正在训练模型，回测步预测开始日期: 2024-11-21
🔄 正在训练模型，回测步预测开始日期: 2024-11-22
🔄 正在训练模型，回测步预测开始日期: 2024-11-25
🔄 正在训练模型，回测步预测开始日期: 2024-11-26
🔄 正在训练模型，回测步预测开始日期: 2024-11-27
🔄 正在训练模型，回测步预测开始日期: 2024-11-28
🔄 正在训练模型，回测步预测开始日期: 2024-11-29
🔄 正在训练模型，回测步预测开始日期: 2024-12-02
🔄 正在训练模型，回测步预测开始日期: 2024-12-03
🔄 正在训练模型，回测步预测开始日期: 2024-12-04
🔄 正在训练模型，回测步预测开始日期: 2024-12-05
🔄 正在训练模型，回测步预测开始日期: 2024-12-06
🔄 正在训练模型，回测步预测开始日期: 2024-12-09
🔄 正在训练模型，回测步预测开始日期: 2024-12-10
🔄 正在训练模型，回测步预测开始日期: 2024-12-11
🔄 正在训练模型，回测步预测开始日期: 2024-12-12
🔄 正在训练模型，回测步预测开始日期: 2024-12-13
🔄 正在训练模型，回测步预测开始日期: 2024-12-16
🔄 正在训练模型，回测步预测开始日期: 2024-12-17
🔄 正在训练模型，回测步预测开始日期: 2024-12-18
🔄 正在训练模型，回测步预测开始日期: 2024-12-19
🔄 正在训练模型，回测步预测开始日期: 2024-12-20
🔄 正在训练模型，回测步预测开始日期: 2024

KeyboardInterrupt: 

In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")

# Define the base directory for saving results and models in Google Drive
drive_output_base_dir = "/content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results"
os.makedirs(drive_output_base_dir, exist_ok=True)
print(f"Base output directory in Google Drive: {drive_output_base_dir}")

Google Drive already mounted.
Base output directory in Google Drive: /content/drive/MyDrive/Colab Notebooks/ant_data/backtest_results


In [ ]:
# Modify the run_backtest function to save the trained model
def run_backtest(stock_code, stock_data, model_builder_func, window_size, pred_size, model_dir="./models", output_dir="./charts", epochs=20):
    """
    Runs a rolling window backtest for a given stock.

    Args:
        stock_code (str): The stock code.
        stock_data (dict): Dictionary containing df, df_scaled, etc. for the stock.
        model_builder_func (function): Function to build a new model.
        window_size (int): The size of the look-back window for training.
        pred_size (int): The size of the prediction window.
        model_dir (str): Directory to save/load models.
        output_dir (str): Directory to save charts and results.
        epochs (int): Number of training epochs per step.

    Returns:
        dict: Dictionary containing collected predicted and actual values and dates.
    """
    df = stock_data["df"]
    df_scaled = stock_data["df_scaled"]
    target_regression_data = stock_data["target_regression_data"]
    target_classification_data = stock_data["target_classification_data"]
    scaler = stock_data["scaler"]
    actual_regression_target_names = stock_data["actual_regression_target_names"]

    # Determine the start and end indices for backtesting
    # We need enough data for the first training window and the first prediction window
    if len(df_scaled) < window_size + pred_size:
        print(f"⚠️ 数据不足以进行回测 ({window_size} + {pred_size})，跳过回测：{stock_code}")
        return None

    # The backtest starts from the first date where we can make a prediction
    # This is after the initial training window (window_size) has passed.
    # The loop will iterate through each day that will be the start of a prediction window.
    backtest_start_index = window_size
    backtest_end_index = len(df_scaled) - pred_size # The last possible start date for a full prediction window

    print(f"🏃‍♂️ 开始对股票 {stock_code} 进行滚动回测...")
    print(f"回测数据范围: {df.index[backtest_start_index]} to {df.index[backtest_end_index]}")

    all_predicted_regression = []
    all_actual_regression = []
    all_predicted_classification = []
    all_actual_classification = []
    all_dates = []

    # Define the directory to save models for this specific window/pred size combination within Drive
    drive_model_sub_dir = os.path.join(drive_output_base_dir, f"window_{window_size}_pred_{pred_size}", "models")
    os.makedirs(drive_model_sub_dir, exist_ok=True)


    # Iterate through each date that will be the start of the prediction window
    for current_pred_start_index in range(backtest_start_index, backtest_end_index + 1):
        # Define the training window (ends the day before the prediction starts)
        train_start_index = current_pred_start_index - window_size
        train_end_index = current_pred_start_index - 1
        train_data_scaled_window = df_scaled[train_start_index : train_end_index + 1]
        train_target_regression_window = target_regression_data[train_start_index : train_end_index + 1]
        train_target_classification_window = target_classification_data[train_start_index : train_end_index + 1]

        # Define the prediction window
        pred_start_index = current_pred_start_index
        pred_end_index = current_pred_start_index + pred_size
        # Ensure prediction window does not go beyond available data for actual values
        actual_pred_end_index = min(pred_end_index, len(df_scaled))
        prediction_indices = range(pred_start_index, actual_pred_end_index)

        # Skip if the prediction window is empty or too short
        if not prediction_indices or len(prediction_indices) < pred_size:
             # This check might not be necessary with the loop range calculation, but added for safety
             print(f"⚠️ 预测窗口太短，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             continue

        # Prepare the single training sample for this step
        # The training sample is the sequence of 'window_size' data points
        # and the targets are the next 'pred_size' data points.
        X_train_sample = df_scaled[train_start_index : train_end_index + 1]
        y_train_regression_sample = target_regression_data[pred_start_index : pred_end_index]
        y_train_classification_sample = target_classification_data[pred_start_index : pred_end_index]

        # Reshape samples to match model input/output expectations (batch_size, window_size, features)
        X_train_sample = np.expand_dims(X_train_sample, axis=0)
        y_train_regression_sample = np.expand_dims(y_train_regression_sample, axis=0)
        y_train_classification_sample = np.expand_dims(y_train_classification_sample, axis=0)
        # Ensure classification target has the last dimension as 1
        if y_train_classification_sample.ndim == 2:
             y_train_classification_sample = np.expand_dims(y_train_classification_sample, axis=-1)


        if X_train_sample.shape[1] != window_size or y_train_regression_sample.shape[1] != pred_size or y_train_classification_sample.shape[1] != pred_size:
             print(f"❌ 训练样本形状不匹配，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             print(f"Expected X shape: (1, {window_size}, {df_scaled.shape[1]}), Actual X shape: {X_train_sample.shape}")
             print(f"Expected y_reg shape: (1, {pred_size}, {target_regression_data.shape[1]}), Actual y_reg shape: {y_train_regression_sample.shape}")
             print(f"Expected y_clf shape: (1, {pred_size}, 1), Actual y_clf shape: {y_train_classification_sample.shape}")
             continue


        # Model training/loading logic
        model_path = os.path.join(drive_model_sub_dir, f"{stock_code}_transformer_model_step_{current_pred_start_index}.h5")


        # Train the model on the single sample for this step
        print(f"🔄 正在训练模型，回测步预测开始日期: {df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
        try:
            # Use the model_builder_func to get a fresh model or load an existing one
            # For simplicity in this step, let's train from scratch at each step.
            # A more advanced backtest would load and potentially update.
            model = build_transformer_model((window_size, df_scaled.shape[1]), target_regression_data.shape[1], pred_size)
            model.fit(X_train_sample,
                      {"regression_output": y_train_regression_sample, "classification_output": y_train_classification_sample},
                      epochs=epochs,
                      batch_size=1, # Train on a single sample
                      verbose=0)

            # Save the model after training for this step
            try:
                model.save(model_path)
                print(f"✅ 模型已保存至：{model_path}")
            except Exception as save_error:
                print(f"❌ 模型保存失败：{model_path} → {save_error}")

        except Exception as e:
             print(f"❌ 模型训练失败，跳过当前回测步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')} → {e}")
             continue


        # Prepare data for prediction (the single sample to predict from)
        # This is the sequence of 'window_size' data points ending the day before the prediction starts.
        predict_data_scaled = df_scaled[current_pred_start_index - window_size : current_pred_start_index]

        # Ensure predict_data_scaled has the correct shape (window_size, num_features)
        if predict_data_scaled.shape[0] != window_size or predict_data_scaled.shape[1] != df_scaled.shape[1]:
             print(f"❌ 预测输入数据维度错误，跳过预测：{df.index[current_pred_start_index].strftime('%Y-%m-%d')}")
             continue

        # Make prediction
        try:
            predictions = model.predict(np.expand_dims(predict_data_scaled, axis=0), verbose=0) # Predict on a single sample
            regression_prediction = predictions[0][0] # Shape: (pred_size, num_regression_targets)
            classification_prediction = predictions[1][0] # Shape: (pred_size, 1)
        except Exception as e:
            print(f"❌ 模型预测失败，跳过当前回测试步：{df.index[current_pred_start_index].strftime('%Y-%m-%d')} → {e}")
            continue

        # Store actual values and dates for the prediction window
        actual_regression_window = target_regression_data[pred_start_index : actual_pred_end_index]
        actual_classification_window = target_classification_data[pred_start_index : actual_pred_end_index]
        prediction_dates = df.index[pred_start_index : actual_pred_end_index]

        # Store the results, only for the dates where actual data is available
        all_predicted_regression.extend(regression_prediction[:len(prediction_indices)])
        all_actual_regression.extend(actual_regression_window)
        # Extend with individual elements from the prediction window
        all_predicted_classification.extend(classification_prediction[:len(prediction_indices)].flatten())
        all_actual_classification.extend(actual_classification_window.flatten())
        all_dates.extend(prediction_dates.tolist())


    # Concatenate results and handle potential overlaps (take the prediction closest to the date)
    if not all_dates:
        print(f"⚠️ 未生成任何回测结果，跳过结果保存：{stock_code}")
        return None

    # Ensure actual and predicted data have consistent shapes before creating DataFrame
    # They should be lists of arrays/lists of size num_targets for regression and 1 for classification per date
    # Let's flatten the lists of arrays for easier DataFrame creation
    flat_predicted_regression = [item for sublist in all_predicted_regression for item in sublist]
    flat_actual_regression = [item for sublist in all_actual_regression for item in sublist]
    # Directly convert to numpy arrays and reshape
    flat_predicted_classification = np.array(all_predicted_classification).reshape(-1, 1)
    flat_actual_classification = np.array(all_actual_classification).reshape(-1, 1)


    results_df = pd.DataFrame({
        "Date": all_dates,
        "Predicted_Regression": list(flat_predicted_regression), # Store as list of arrays/lists
        "Actual_Regression": list(flat_actual_regression),
        "Predicted_Classification": list(flat_predicted_classification),
        "Actual_Classification": list(flat_actual_classification)
    })

    results_df["Date"] = pd.to_datetime(results_df["Date"])
    results_df.set_index("Date", inplace=True)
    results_df.sort_index(inplace=True)

    # Handle potential duplicate dates by keeping the last prediction for each date
    # This is a simplification; a proper backtest would handle multi-step predictions
    # and their validity over time more carefully.
    results_df = results_df[~results_df.index.duplicated(keep='last')].copy()


    print(f"✅ 滚动回测对股票 {stock_code} 完成.")

    return results_df

# Now, adapt the main loop to use run_backtest
# We will loop through stock codes and then call run_backtest for each combination of window/pred size
results = {} # Reset results to store backtest results

# Loop through stock codes first
for code in stock_codes:
    if code not in all_stocks_data:
        print(f"跳过股票 {code}，回测数据未加载。")
        continue

    stock_data = all_stocks_data[code]
    print(f"\n--- 对股票 {code} 进行窗口组合测试 ---")

    # Loop through different window size combinations
    for current_window_size in window_sizes_to_try:
        for current_pred_size in pred_sizes_to_try:
            print(f"\n--- 测试组合: 回溯窗口={current_window_size}, 预测窗口={current_pred_size} ---")

            # Define the local charts directory for this combination
            local_charts_sub_dir = f"./charts/window_{current_window_size}_pred_{current_pred_size}"
            os.makedirs(local_charts_sub_dir, exist_ok=True)

            # Run the backtest for the current stock and window combination
            # Note: The run_backtest function will save models to Drive within the loop
            backtest_results_df = run_backtest(
                stock_code=code,
                stock_data=stock_data,
                model_builder_func=build_transformer_model, # Pass the function
                window_size=current_window_size,
                pred_size=current_pred_size,
                model_dir="./models", # This local dir is less relevant now
                output_dir=local_charts_sub_dir, # Pass the local charts dir
                epochs=1 # Reduced epochs significantly for testing this iteration
            )

            if backtest_results_df is not None:
                # Store results for the current combination
                combination_key = f"{code}_window_{current_window_size}_pred_{current_pred_size}"
                results[combination_key] = backtest_results_df

                # Save raw backtest results to CSV in Google Drive
                drive_results_sub_dir = os.path.join(drive_output_base_dir, f"window_{current_window_size}_pred_{current_pred_size}", "results")
                os.makedirs(drive_results_sub_dir, exist_ok=True)
                results_path_drive = os.path.join(drive_results_sub_dir, f"{code}_backtest_results.csv")
                backtest_results_df.to_csv(results_path_drive)
                print(f"✅ 回测结果CSV已保存至 Google Drive：{results_path_drive}")

                # Copy local charts directory to Google Drive
                drive_charts_sub_dir = os.path.join(drive_output_base_dir, f"window_{current_window_size}_pred_{current_pred_size}", "charts")
                os.makedirs(drive_charts_sub_dir, exist_ok=True)

                # Use shutil to copy the directory contents
                import shutil
                try:
                    # Copy contents of local_charts_sub_dir to drive_charts_sub_dir
                    # Ensure we copy files and subdirectories
                    for item in os.listdir(local_charts_sub_dir):
                        s = os.path.join(local_charts_sub_dir, item)
                        d = os.path.join(drive_charts_sub_dir, item)
                        if os.path.isdir(s):
                            shutil.copytree(s, d, dirs_exist_ok=True)
                        else:
                            shutil.copy2(s, d)
                    print(f"✅ 本地图表已复制到 Google Drive：{drive_charts_sub_dir}")
                except Exception as copy_error:
                    print(f"❌ 复制图表到 Google Drive 失败：{copy_error}")


# After all backtests are run, you can analyze the 'results' dictionary
print("\n--- 所有回测任务完成 ---")
# Further steps would involve analyzing the 'results' dictionary, calculating metrics, and visualizing.

NameError: name 'stock_codes' is not defined